In [2]:
import os
import numpy as np

midi_file_paths = []
for root, dirs, files in os.walk("POP909-Dataset/POP909"):
    for file in files:
        if file.endswith(".mid") and not root.endswith("versions"):
            midi_file_paths.append(os.path.join(root, file))

print(f"\nTotal MIDI files found: {len(midi_file_paths)}")


Total MIDI files found: 909


In [3]:
import mido

def ExtractMelodyTones(file_path):

    abs_path = file_path

    midi_mel = mido.MidiFile(abs_path)
    seq = []

    for msg in midi_mel.tracks[1]: #Vi vil kun have melodistemen
        if msg.type == 'note_on' and msg.velocity > 0:  # Kun note_on with velocity > 0
            if seq == []:
                seq.append(msg.note)
            elif msg.note != seq[-1]:
                seq.append(msg.note)

    return seq

def ConvertToFreqs(seq):
    return [round(440 * (2**((tone - 69)/12)), 3) for tone in seq]

melodies_midi = [ExtractMelodyTones(path) for path in midi_file_paths]
melodies_freqs = [ConvertToFreqs(seq) for seq in melodies_midi]

In [4]:
print("There is " + str(len(melodies_freqs)) + " sequences")
print("with an average length of " + str(sum([len(seq) for seq in melodies_freqs]) / len(melodies_freqs)) + " tones")

individual_tones = set()
for seq in melodies_midi:
    for tone in seq:
        individual_tones.add(tone)

print("Consisting of " + str(len(individual_tones)) + " unique tones")

There is 909 sequences
with an average length of 260.1771177117712 tones
Consisting of 55 unique tones


### Variable Order Markov Model

First we need the n-grams the maximum order of seven (septogram)

In [5]:
n_grams = []
max_order = 7

for i in range(max_order+1):
    i_gram = []
    if i == 0:
        for seq in melodies_midi:
            for tone in seq:
                i_gram.append(tuple([tone]))
    else:
        for seq in melodies_midi:
            for j, tone in enumerate(seq[:-i]):
                i_gram.append(tuple(seq[j:j+i+1]))

    n_grams.append(i_gram)



In [6]:
for i, n_gram in enumerate(n_grams):
    print("There is " + str(len(n_gram)) + " " + str(i) + "-grams in the corpus")

There is 236501 0-grams in the corpus
There is 235592 1-grams in the corpus
There is 234683 2-grams in the corpus
There is 233774 3-grams in the corpus
There is 232865 4-grams in the corpus
There is 231956 5-grams in the corpus
There is 231047 6-grams in the corpus
There is 230138 7-grams in the corpus


In [7]:
from collections import defaultdict, Counter

n_gram_counts = [defaultdict(int) for _ in range(8)]

for order, i_grams in enumerate(n_grams):
    for gram in i_grams:
        n_gram_counts[order][gram] += 1 

Calculates the probabilities

$$
p_{esc}=number of distinct next symbols/total count for this context​
$$

In [8]:
def GetSurprisePool(probs, surprisal, exclude=None, probe=None):
    filtered = {t: p for t, p in probs.items()}
    
    if exclude:
        filtered = {t: p for t, p in filtered.items() if t != exclude}

    if probe:
        filtered = {t: p for t, p in filtered.items() if t < probe + 10 and t > probe - 10}

    if not filtered:
        return {}

    # Convert probabilities to surprisal values: S(t) = -log2(p(t))
    surprisal_vals = {t: -math.log2(p) for t, p in filtered.items() if p > 0}

    # Separate "seen" tones (lower surprisal bound) from unseen noise floor
    min_surprisal = min(surprisal_vals.values())
    max_surprisal = max(surprisal_vals.values())
    surprisal_range = max_surprisal - min_surprisal

    # Sort by surprisal ascending (low surprisal = expected; high surprisal = unexpected)
    seen = sorted(surprisal_vals.items(), key=lambda x: x[1])

    if len(seen) < 2:
        seen = sorted(surprisal_vals.items(), key=lambda x: x[1])

    cutoff = max(2, len(seen) // 4)

    if surprisal:
        # High-surprisal pool: avoid the very tail (potential outliers).
        # Take the 2nd quartile from the top instead of the absolute highest.
        # i.e., skip the top ~25% most surprising, use the next 25% above median.
        upper_end = max(cutoff * 2, len(seen) - 1)
        lower_end = max(cutoff, upper_end - cutoff)
        pool_items = seen[lower_end:upper_end]
        if len(pool_items) < 2:
            pool_items = seen[-(cutoff + 1):-1] if len(seen) > cutoff else seen[-cutoff:]
    else:
        # Low-surprisal pool: most expected tones (lowest surprisal = top probabilities)
        pool_items = seen[:cutoff]

    pool = {t: filtered[t] for t, _ in pool_items}
    return pool


In [9]:
import random
import math

def PredictNext(seq, exclude=None):
    probs = defaultdict(float)
    remaining_prob = 1.0
    order = min(len(seq), 7)

    for o in range(order, 0, -1):
        context = seq[order - o :]
        total = sum(
            n_gram_counts[o][context + (s,)] for s in individual_tones
        )
        if total == 0:
            continue
        distinct = sum(
            1 for s in individual_tones if n_gram_counts[o][context + (s,)] > 0
        )
        p_esc = distinct / (total + distinct)

        for s in individual_tones:
            count = n_gram_counts[o][context + (s,)]
            if count > 0:
                probs[s] += remaining_prob * (1 - p_esc) * (count / total)
        
        remaining_prob *= p_esc

    unseen_prob = remaining_prob / len(individual_tones)
    for s in individual_tones:
        probs[s] += unseen_prob

    probs.pop(seq[-1], None)
    if exclude:
        probs.pop(exclude, None)

    entropy = -sum(p * math.log2(p) for p in probs.values() if p > 0)

    new_keys = dict(filter(lambda x: x[0] < seq[-1]+10 and x[0] > seq[-1]-10, probs.items()))
    if not new_keys:
        return False
    real = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    real_prob = probs[real]
    probs.pop(real, None)

    return real, real_prob, entropy


Generating a simple melodi with pure probability

In [10]:
def ToneRandChoice(probs):
    total = sum(probs.values())
    choice = np.random.random() * total
    for key, value in probs.items():
        if choice < value:
            return key, value
        choice -= value


def GenerateMelody(probe, length):
    context = (probe,)
    all_probs = [1]
    entro = [0]
    for i in range(length):
        mel = PredictNext(context)
        if not mel:
            return False
        new_tone, prob, entropy = mel
        all_probs.append(prob)
        context += (new_tone,)
        entro.append(entropy)
    return list(context), all_probs, entro


Using this to create binary trees for sequential melody production
$$
N=2^0+2^1+...+2^{l-1}=2^l-1
$$

In [11]:
import math

def GenerateBinaryTree(probe, length):
    tree = [0 for _ in range(2**length-1)]
    prob_tree = [0 for _ in range(2**length-1)]
    entropy_tree = [0 for _ in range(2**length-1)]
    pitch_dif_tree = [0 for _ in range(2**length-1)]
    tree[0] = probe
    prob_tree[0] = 1

    for layer in range(1,length,1):
        fst_pos = 2**layer - 1
        for node in range(0,2**layer, 2):
            parents = [math.ceil((node+fst_pos)/2)-1]
            for i in range(layer-2):
                parents.append(math.ceil(parents[-1]/2) - 1)
            context = tuple([tree[i] for i in parents][::-1])

            mel = PredictNext(context)
            if not mel:
                return False
            tone1, prob1, entropy = mel

            mel = PredictNext(context, exclude=tone1)
            if not mel:
                return False
            tone2, prob2, _ = mel
            
            choice = np.random.random()
            if choice > 0.5:
                tone1, tone2 = tone2, tone1
                prob1, prob2 = prob2, prob1
            
            tree[fst_pos + node] = tone1
            tree[fst_pos + node + 1] = tone2

            prob_tree[fst_pos + node] = prob1
            prob_tree[fst_pos + node + 1] = prob2

            entropy_tree[fst_pos + node] = entropy
            entropy_tree[fst_pos + node + 1] = entropy

            pitch_dif1 = tone1 - context[-1]
            pitch_dif2 = tone2 - context[-1]

            pitch_dif_tree[fst_pos + node] = pitch_dif1
            pitch_dif_tree[fst_pos + node + 1] = pitch_dif2

    return tree, prob_tree, entropy_tree, pitch_dif_tree

Just for testing the tree, i will make a function to choose a random sequence

In [12]:
def TreeRandSeq(tree, length):
    seq = [tree[0]]
    parent = 0
    for i in range(length-1):
        choice = np.random.random()
        if choice > 0.5:
            seq.append(tree[2*(parent+1)-1])
            parent = 2*(parent+1)-1
        else:
            seq.append(tree[2*(parent+1)])
            parent = 2*(parent+1)
    return seq

Generating a melody with change
- Input is probe, position and suprise-factor

In [13]:
import random

def PredictNextWithAlternative(seq, surprise):
    probs = defaultdict(float)
    remaining_prob = 1.0
    order = min(len(seq), 7)

    for o in range(order, 0, -1):
        context = seq[order - o :]
        total = sum(
            n_gram_counts[o][context + (s,)] for s in individual_tones
        )
        if total == 0:
            continue
        distinct = sum(
            1 for s in individual_tones if n_gram_counts[o][context + (s,)] > 0
        )
        p_esc = distinct / (total + distinct)

        for s in individual_tones:
            count = n_gram_counts[o][context + (s,)]
            if count > 0:
                probs[s] += remaining_prob * (1 - p_esc) * (count / total)
        
        remaining_prob *= p_esc

    unseen_prob = remaining_prob / len(individual_tones)
    for s in individual_tones:
        probs[s] += unseen_prob

    entropy = -sum(p * math.log2(p) for p in probs.values() if p > 0)

    probs.pop(seq[-1], None)

    surprisal1, surprisal2 = surprise

    new_keys = GetSurprisePool(probs, surprisal1, seq[-1])
    if not new_keys:
        return False
    real = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    real_prob = probs[real]
    probs.pop(real, None)

    new_keys = GetSurprisePool(probs, surprisal2)
    if not new_keys:
        return False
    alt = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    alt_prob = probs[alt]

    return (real, real_prob), (alt, alt_prob), entropy


def GenerateMelodyWithChange(probe, length, pos, surprise):
    context = (probe,)
    probs = [1]
    entro = [0]
    for i in range(length-1):
        if i+1 == pos:
            pred = PredictNextWithAlternative(context, surprise)
            if not pred:
                return False
            tone, alt, entropy = pred
            new_tone, val = tone
        else:
            pred = PredictNext(context)
            if not pred:
                return False
            new_tone, val, entropy = pred
        context += (new_tone,)
        probs.append(val)
        entro.append(entropy)
    return list(context), probs, alt, entro

mel = GenerateMelodyWithChange(61, 8, 3, (True, False))
if not mel == False:
    tones, probs, alts, entro = mel
    print(tones)
    print(probs)
    print(alts)
    print(entro)
else:
    print("Error")

[61, 59, 56, 92, 91, 87, 89, 91]
[1, 0.1734813183089045, 0.1423554829503072, 2.4605491945802306e-06, 0.1475378787878788, 0.19583333333333333, 0.8216644987233223, 0.9573101658533312]
(61, 0.14898051245037894)
[0, 3.408827837528595, 2.7889517297253446, 2.499261486125245, 3.387866422869959, 1.3109490862028104, 0.8478213731806808, 0.3506644043865643]


Now the same for trees

In [14]:
def PredictTwoNextWithAlternative(seq, surprise):
    probs = defaultdict(float)
    remaining_prob = 1.0
    order = len(seq)

    for o in range(order, 0, -1):
        context = seq[order - o :]
        total = sum(
            n_gram_counts[o][context + (s,)] for s in individual_tones
        )
        if total == 0:
            continue
        distinct = sum(
            1 for s in individual_tones if n_gram_counts[o][context + (s,)] > 0
        )
        p_esc = distinct / (total + distinct)

        for s in individual_tones:
            count = n_gram_counts[o][context + (s,)]
            if count > 0:
                probs[s] += remaining_prob * (1 - p_esc) * (count / total)
        
        remaining_prob *= p_esc

    unseen_prob = remaining_prob / len(individual_tones)
    for s in individual_tones:
        probs[s] += unseen_prob
    
    entropy = -sum(p * math.log2(p) for p in probs.values() if p > 0)

    probs.pop(seq[-1], None)

    surprisal1, surprisal2 = surprise

    new_keys = GetSurprisePool(probs, surprisal1, probe=seq[-1])
    if not new_keys:
        return False
    real1 = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    real1_prob = probs[real1]

    probs.pop(real1, None)

    new_keys = GetSurprisePool(probs, surprisal1, probe=seq[-1], exclude=real1)
    if not new_keys:
        return False

    real2 = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    real2_prob = probs[real2]

    probs.pop(real1, None)
    probs.pop(real2, None)

    new_keys = GetSurprisePool(probs, surprisal2, probe=seq[-1])
    if not new_keys:
        return False
    alt = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    alt_prob = probs[alt]

    return (real1, real1_prob), (real2, real2_prob), (alt, alt_prob), entropy

def GenerateBinaryTreeWithChange(probe, length, pos, surprise):
    tree = [0 for _ in range(2**length-1)]
    prob_tree = [0 for _ in range(2**length-1)]
    ent_tree = [0 for _ in range(2**length-1)]
    pitch_dif_tree = [0 for _ in range(2**length-1)]
    alts = []

    tree[0] = probe
    prob_tree[0] = 1
    for layer in range(1,length,1):
        if layer+1 == pos:
            fst_pos = 2**layer - 1
            for node in range(0,2**layer, 2):
                parents = [math.ceil((node+fst_pos)/2)-1]
                for i in range(layer-2):
                    parents.append(math.ceil(parents[-1]/2) - 1)

                context = tuple([tree[i] for i in parents][::-1])
                pred = PredictTwoNextWithAlternative(context, surprise)
                if not pred:
                    return False
                real1, real2, alt, entropy = pred
                if real1 == False or real2 == False or alt == False:
                    return False
                tone1, prob1 = real1
                tone2, prob2 = real2
                tree[fst_pos + node] = tone1
                tree[fst_pos + node + 1] = tone2
                alts.append(alt)

                prob_tree[fst_pos + node] = prob1
                prob_tree[fst_pos + node + 1] = prob2

                ent_tree[fst_pos + node] = entropy
                ent_tree[fst_pos + node + 1] = entropy

                pitch_dif1 = tone1 - context[-1]
                pitch_dif2 = tone2 - context[-1]

                pitch_dif_tree[fst_pos + node] = pitch_dif1
                pitch_dif_tree[fst_pos + node + 1] = pitch_dif2
        else:
            fst_pos = 2**layer - 1
            for node in range(0,2**layer, 2):
                parents = [math.ceil((node+fst_pos)/2)-1]
                for i in range(layer-2):
                    parents.append(math.ceil(parents[-1]/2) - 1)
                context = tuple([tree[i] for i in parents][::-1])
                
                mel = PredictNext(context)
                if not mel:
                    return False
                tone1, prob1, entropy = mel

                mel = PredictNext(context, exclude=tone1)
                if not mel:
                    return False
                tone2, prob2, _ = mel

                choice = np.random.random()
                if choice > 0.5:
                    tone1, tone2 = tone2, tone1
                    prob1, prob2 = prob2, prob1
                tree[fst_pos + node] = tone1
                tree[fst_pos + node + 1] = tone2

                prob_tree[fst_pos + node] = prob1
                prob_tree[fst_pos + node + 1] = prob2

                ent_tree[fst_pos + node] = entropy
                ent_tree[fst_pos + node + 1] = entropy

                pitch_dif1 = tone1 - context[-1]
                pitch_dif2 = tone2 - context[-1]

                pitch_dif_tree[fst_pos + node] = pitch_dif1
                pitch_dif_tree[fst_pos + node + 1] = pitch_dif2

    return tree, prob_tree, alts, ent_tree, pitch_dif_tree

mel = GenerateBinaryTreeWithChange(61, 5, 2, (True, False))
if not mel == False:
    tones, probs, alts, entro, pitch_dif = mel
    print(tones)
    print(probs)
    print(alts)
    print(entro)
    print(pitch_dif)
else:
    print("Error")

[61, 67, 54, 69, 64, 52, 59, 72, 64, 67, 62, 54, 57, 61, 57, 70, 74, 67, 62, 72, 64, 57, 64, 57, 55, 54, 59, 64, 63, 55, 59]
[1, 0.0016888361715947922, 0.008833912282188144, 0.2655477557657546, 0.09306512380855589, 0.03453869550523826, 0.19810746873943902, 0.19238288447961077, 0.050840522039287794, 0.26675148523824005, 0.3418077969203062, 0.35420531849103276, 0.2657699443413729, 0.4862884875194204, 0.05114358063088188, 0.027982320678266035, 0.25196262665525887, 0.27259099449956786, 0.4485127496410387, 0.07697394719609113, 0.17724472317711007, 0.03610748125984707, 0.4423491386255809, 0.3510130513847986, 0.0768937382320282, 0.8134509017510748, 0.09441255111670337, 0.026416799529652195, 0.6637421970274635, 0.7505816535251493, 0.10619864142318385]
[(59, 0.1734813183089045)]
[0, 3.410569604072596, 3.410569604072596, 3.360822132241086, 3.360822132241086, 3.6923268684716417, 3.6923268684716417, 2.865941993033453, 2.865941993033453, 2.751271202698945, 2.751271202698945, 2.6169872085958454, 2.6

The actual generation of sequences

In [15]:
import pandas as pd

sur_factors = [
    (True, True),
    (True, False),
    (False, True),
    (False, False)
]

pos_factors = [
    (2,3),
    (4,5),
    (6,7)
]

change_factors = [
    True,
    False
]

gen_factors = [
    True,
    False
]

probe_tones = [
    60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74
]

length = 8

data = {
    "Generated": [],
    "Change": [],
    "Position": [],
    "Surprisal": [],
    "PitchDif": [],
    "Entropy": [],
    "Probe": [],
    "Sequence": [],
    "Probabilites": [],
    "Alternatives": []
}

for gen in gen_factors:
    for change in change_factors:
        for pos in pos_factors:
            for surprise in sur_factors:
                probe = random.choice(probe_tones)
                if change:
                    position = random.choice(pos)

                    mel = GenerateBinaryTreeWithChange(probe, length, position, surprise)
                    while mel == False:
                        probe = random.choice(probe_tones)
                        mel = GenerateBinaryTreeWithChange(probe, length, position, surprise)
                    seq, probs, alt, entro, pitch_dif = mel
                else:
                    position = -1
                    surprise = False

                    mel = GenerateBinaryTree(probe, length)
                    while mel == False:
                        probe = random.choice(probe_tones)
                        mel = GenerateBinaryTree(probe, length)
                    seq, probs, entro, pitch_dif = mel
                    alt = False

                data["Generated"].append(gen)
                data["Change"].append(change)
                data["Position"].append(position)
                data["Surprisal"].append(surprise)
                data["PitchDif"].append(pitch_dif)
                data["Entropy"].append(entro)
                data["Probe"].append(probe)
                data["Sequence"].append(seq)
                data["Probabilites"].append([-math.log2(p) for p in probs])
                data["Alternatives"].append(alt)

                print("Generated sequence")

df = pd.DataFrame(data)
df.to_csv("sequences_new.csv", index=False)

Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence
Generated sequence


In [ ]:
def GeneratePractice():
    length = 8

    data = {
        "Generated": [],
        "Change": [],
        "Position": [],
        "Surprisal": [],
        "PitchDif": [],
        "Entropy": [],
        "Probe": [],
        "Sequence": [],
        "Probabilites": [],
        "Alternatives": []
    }

    probe = 65
    position = -1
    surprise = False
    mel = GenerateBinaryTree(probe, length)
    while mel == False:
        mel = GenerateBinaryTree(probe, length)
    seq, probs, entro, pitch_dif = mel
    alt = False

    data["Generated"].append(gen)
    data["Change"].append(change)
    data["Position"].append(position)
    data["Surprisal"].append(surprise)
    data["PitchDif"].append(pitch_dif)
    data["Entropy"].append(entro)
    data["Probe"].append(probe)
    data["Sequence"].append(seq)
    data["Probabilites"].append([-math.log2(p) for p in probs])
    data["Alternatives"].append(alt)

    print("Generated sequence")

    probe = 68
    position = -1
    surprise = False

    mel = GenerateBinaryTree(probe, length)
    while mel == False:
        probe = random.choice(probe_tones)
        mel = GenerateBinaryTree(probe, length)
    seq, probs, entro, pitch_dif = mel
    alt = False

    data["Generated"].append(gen)
    data["Change"].append(change)
    data["Position"].append(position)
    data["Surprisal"].append(surprise)
    data["PitchDif"].append(pitch_dif)
    data["Entropy"].append(entro)
    data["Probe"].append(probe)
    data["Sequence"].append(seq)
    data["Probabilites"].append([-math.log2(p) for p in probs])
    data["Alternatives"].append(alt)

    print("Generated sequence")

    df = pd.DataFrame(data)
    df.to_csv("practice_sequences_new.csv", index=False)

GeneratePractice()

Generated sequence
Generated sequence
